# TensorFlow tutorial 


# Import the libraries 

In [1]:
# To silence the TensorFlow warnings, you can use the following code before you import the TensorFlow library.
import os
os.environ['TF_CPP_MIN_LOG_LEVEL'] = '3'

import numpy as np
import tensorflow as tf
from tensorflow.keras.preprocessing.text import Tokenizer
from tensorflow.keras import layers
from tensorflow.keras import losses
import re
import string
import matplotlib.pyplot as plt

print("Imports successful!")

Imports successful!


In [2]:
seed = 42

# sets the global random seed 
np.random.seed(seed) 
tf.random.set_seed(seed)

print (f"Random seed set to {seed}")

Random seed set to 42


## Load the data

In [3]:
data_dir = r"D:\courses\1_AI_systems\03_resources\datasets\Coursera\Natural_Language_Processing_with_Sequence_Models\Lab_1\aclImdb"

In [4]:
# Create the training set. Use 80% of the data
raw_training_set = tf.keras.utils.text_dataset_from_directory(
    f'{data_dir}/train',
    labels = 'inferred',
    label_mode = 'int',
    batch_size=32,
    validation_split =0.2,
    subset = 'training',
    seed = seed 
)

# Create the validation set. Use 20% of the data that was not used for training
raw_Validation_set = tf.keras.utils.text_dataset_from_directory(
    f'{data_dir}/train',
    labels = 'inferred',
    label_mode = 'int',
    batch_size=32,
    validation_split =0.2,
    subset = 'validation',
    seed = seed 
)

# Create the test set
raw_test_set = tf.keras.utils.text_dataset_from_directory(
    f'{data_dir}/test',
    labels = 'inferred',
    label_mode = 'int',
    batch_size=32
)


Found 5000 files belonging to 2 classes.
Using 4000 files for training.
Found 5000 files belonging to 2 classes.
Using 1000 files for validation.
Found 5000 files belonging to 2 classes.


In [5]:
print (f"Label 0 corresponds to {raw_training_set.class_names[0]}")
print (f"Label 1 corresponds to {raw_training_set.class_names[1]}")

Label 0 corresponds to neg
Label 1 corresponds to pos


In [6]:
# Take one batch from the dataset and print out the first three datapoints
for text_batch, label_batch in raw_training_set.take(1):
    for i in range (3):
        print (f"Review:\n {text_batch.numpy()[i]}")
        print (f"Label:\n {label_batch.numpy()[i]}\n")

Review:
 b'This is a reunion, a team, and a great episode of Justice. From hesitation to resolution, Clark has made a important leap from a troubled teenager who was afraid of a controlled destiny, to a Superman who, like Green Arrow, sets aside his emotions to his few loved ones, ready to save the whole planet. This is not just a thrilling story about teamwork, loyalty, and friendship; this is also about deciding what\'s more important in life, a lesson for Clark. I do not want the series to end, but I hope the ensuing episodes will strictly stick to what Justice shows without any "rewind" pushes and put a good end here of Smallville---and a wonderful beginning of Superman.<br /><br />In this episode, however, we should have seen more contrast between Lex and the Team. Nine stars should give it enough credit.'
Label:
 1

Review:
 b'"Hey Babu Riba" is a film about a young woman, Mariana (nicknamed "Esther" after a famous American movie star), and four young men, Glenn, Sacha, Kicha, an

## Prepare the data

In [ ]:
# Set the max number of words and sequence length
max_features = 10000
sequence_length = 250 # used in vectorize_layer and model_sequential

# Define the custom standardization function
def custom_standardization (input_data):
    # Convert all text to lowercase
    lowercase = tf.strings.lower(input_data)
    # Remove HTML tags
    stripped_html = tf.strings.regex_replace(lowercase, '<br />', ' ')
    # Remove punctuation 
    replaced = tf.strings.regex_replace(stripped_html, '[%s]' % re.escape (string.punctuation),'' )
    return replaced

# Create a layer that can be used to convert text to vectors 
vectorize_layer = layers.TextVectorization(
    standardize = custom_standardization,
    max_tokens = max_features,
    output_mode = 'int',
    output_sequence_length = sequence_length
)

In [8]:
# Build the vocabulary 
train_text = raw_training_set.map(lambda x, y: x)
vectorize_layer.adapt(train_text)

# Print out the vocabulary size
print (f'Vocabulary size: {len(vectorize_layer.get_vocabulary())}')

Vocabulary size: 10000


In [9]:
# Define the final function that will be used to vectorize the text.
def vectorize_text(text, label): 
    text = tf.expand_dims(text, -1)
    return vectorize_layer (text), label 

# Get one batch and select the first datapoint 
text_batch, label_batch = next(iter(raw_test_set))
first_review, first_label = text_batch[0], label_batch[0]

# Show the raw data
print(f'Review:\n{first_review}')
print(f'\nlabel: {raw_training_set.class_names[first_label]}')

# Show the vectorized data
print(f'\nVectorized review\n {vectorize_text(first_review, first_label)}')

Review:
b"A coach who used to be good, but has had to move to a new area to get away from what he once did; a team of no hopers; the star player who has a row with the coach; the assistant coach with an alcohol problem, the little guy who gets kicked around scoring the winning basket; the whole town against the coach (all except for the leading lady). It's the old story - right out of the British comic books, where a team of speccy geeks and fatties take on the Brazilians at soccer and win.<br /><br />This film, admittedly well put together, is full of these clich\xc3\xa9s. The only part I didn't predict was when at the town meeting, the star player suddenly decided to side with the coach. No reason was ever given. It seemed like a very strange thing to do seeing that everything was going his way hitherto. Maybe he just felt sorry for the poor guy.<br /><br />Gene Hackman plays a locker room Popey Doyle. I am sure that an actor of Dennis Hopper's calibre found nothing particularly chal

In [ ]:
# Apply the vectorization function to vectorize all three datasets.
train_ds = raw_training_set.map(vectorize_text)
val_ds = raw_Validation_set.map(vectorize_text)
test_ds = raw_test_set.map(vectorize_text)

### Configure the Dataset

There are two important methods that you should use when loading data to make sure that I/O does not become blocking.

`.cache()` keeps data in memory after it's loaded off disk. This will ensure the dataset does not become a bottleneck while training your model. If your dataset is too large to fit into memory, you can also use this method to create a performant on-disk cache, which is more efficient to read than many small files.

`.prefetch()` overlaps data preprocessing and model execution while training.

In [11]:
AUTOTUNE = tf.data.AUTOTUNE

train_ds = train_ds.cache().prefetch(buffer_size = AUTOTUNE)
test_ds = test_ds.cache().prefetch(buffer_size = AUTOTUNE)

## Create a sequential model
**Sequential model**: simple stack of layers where each layer has exactly one input tensor and one output tensor

In [23]:
embedding_dim = 16

# Create the model by calling tf.keras.sequential, where the layers are given in a list.
model_sequential = tf.keras.Sequential([
    layers.Input(shape=(sequence_length,)),
    layers.Embedding(max_features, embedding_dim),
    layers.GlobalAveragePooling1D(),
    layers.Dense(1, activation='sigmoid')
])

model_sequential.summary()

Model: "sequential_4"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding_4 (Embedding)         │ (None, 250, 16)        │       160,000 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_4      │ (None, 16)             │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_4 (Dense)                 │ (None, 1)              │            17 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 160,017 (625.07 KB)

 Trainable params: 160,017 (625.07 KB)

 Non-trainable params: 0 (0.00 B)

In [20]:
model_sequential.compile(loss= losses.BinaryCrossentropy(),
                        optimizer = 'adam',
                        metrics = ['accuracy'])

## Create a model using functional API

In [ ]:
# Define the inputs
inputs = tf.keras.Input(shape=(None,))

# Define the first layers
embedding = layers.Embedding(max_features, embedding_dim)



## Train the model

In [ ]:
# Select teh model to be used and trained. The results should be the same.
# model = model_functional